# Before/After Reel — Deep Analysis Reproducibility Notebook

Bu notebook v4.0 consumer-utility skorunu, review coverage kontrollerini ve pre-launch finans senaryolarını yerel structured dosyalardan yeniden hesaplar. Rakamlar pazar tahmini değil, rapor tutarlılık kontrolüdür.

## Sources and assumptions

- `data/assessment.json`: formal v4 assessment
- `data/evidence.jsonl`: evidence records
- `data/review-sample.json`: Apple RSS coverage ve themes
- `data/competitor-expanded.json`: selected direct, adjacent ve substitute universe
- `data/financial-scenarios.json`: explicit scenario inputs
- Lifetime price $9.99 ve Apple commission %15 yalnız senaryo varsayımıdır.

In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
OUT = ROOT / 'output/2026-07-18-us-before-after-reel-deep'
assert OUT.exists(), f'Run from repository root: {ROOT}'
assessment = json.loads((OUT / 'data/assessment.json').read_text())
reviews = json.loads((OUT / 'data/review-sample.json').read_text())
competitors = json.loads((OUT / 'data/competitor-expanded.json').read_text())
finance = json.loads((OUT / 'data/financial-scenarios.json').read_text())
evidence = [json.loads(line) for line in (OUT / 'data/evidence.jsonl').read_text().splitlines() if line.strip()]
assert len({row['evidence_id'] for row in evidence}) == len(evidence)
print({'evidence_records': len(evidence), 'idea_id': assessment['idea_id']})

## Opportunity score

Consumer Utility weights are applied to the low/base/high 1–5 dimension values.

In [ ]:
weights = {
    'problem_demand': 25,
    'monetization_economics': 15,
    'competitive_wedge': 20,
    'distribution_access': 25,
    'repeat_value_retention': 5,
    'timing': 10,
}

def round_half_up(value):
    return int(value + 0.5)

def opportunity(bound):
    return round_half_up(sum(weights[key] * assessment['dimensions'][key][bound] / 5 for key in weights))

calculated = {bound: opportunity(bound) for bound in ('low', 'base', 'high')}
reported = {bound: assessment['opportunity'][bound] for bound in ('low', 'base', 'high')}
assert calculated == reported == {'low': 28, 'base': 48, 'high': 66}
assert calculated['low'] <= calculated['base'] <= calculated['high']
print(calculated)

## Competitor landscape consistency

Selected-universe counts validate the report's direct, adjacent and substitute comparisons. Rating counts remain storefront proxies, not installs or revenue.

In [ ]:
direct = competitors['direct_apps']
adjacent = competitors['adjacent_apps']
substitutes = competitors['substitutes']
summary = competitors['computed_summary']
assert len(direct) == summary['direct_apps_scanned'] == 15
assert len(adjacent) == summary['adjacent_apps_scanned'] == 14
assert len(substitutes) == 7
assert sum(row['rating_count'] >= 1000 for row in direct) == summary['direct_apps_with_1000_plus_us_ratings'] == 3
recent_direct = [row for row in direct if row['original_release'] >= '2025-01-01']
assert len(recent_direct) == summary['direct_apps_released_since_2025'] == 7
assert sum(row['rating_count'] == 0 for row in recent_direct) == summary['recent_direct_apps_with_zero_visible_ratings'] == 6
assert summary['adjacent_apps_advertising_auto_alignment'] == 3
assert summary['closest_simple_flow_competitor'].startswith('Before & After Video Maker')
print({'direct': len(direct), 'adjacent': len(adjacent), 'substitutes': len(substitutes), 'recent_direct': len(recent_direct), 'recent_zero_rating': 6})

## Review coverage and theme integrity

Theme counts are multi-label and therefore need not sum to the 11-review denominator.

In [ ]:
recent_coverage = sum(row['recent'] for row in reviews['coverage'])
recent_rows = len(reviews['recent_reviews'])
assert recent_coverage == recent_rows == 11
assert sum(row['recent_3'] for row in reviews['coverage']) == 0
assert sum(row['recent_4_5'] for row in reviews['coverage']) == 0
coded_from_rows = {}
for row in reviews['recent_reviews']:
    for theme in row['themes']:
        coded_from_rows[theme] = coded_from_rows.get(theme, 0) + 1
expected = {
    'monetization_friction': 7,
    'missing_controls_or_source_access': 4,
    'export_framing_resolution': 2,
    'crash_stability': 2,
    'ux_complexity': 2,
    'support_cancellation': 2,
}
assert coded_from_rows == expected
print({'recent_n': recent_rows, 'themes': coded_from_rows})

## Financial scenario recalculation

Every output is recalculated from explicit inputs; this validates arithmetic, not the assumptions.

In [ ]:
price = finance['product_offer_assumption']['price']
commission = finance['product_offer_assumption']['apple_commission_rate']
recalculated = []
for s in finance['scenarios']:
    proceeds = s['installs'] * s['conversion'] * (1 - s['refund_rate']) * price * (1 - commission)
    paid_spend = s['installs'] * s['paid_share'] * s['paid_cpi']
    support = s['purchasers'] * s['ticket_rate'] * s['support_minutes'] / 60 * s['hourly_support_cost']
    contribution = proceeds - paid_spend - support
    assert abs(proceeds - s['developer_proceeds']) < 0.02
    assert abs(paid_spend - s['paid_spend']) < 0.01
    assert abs(support - s['support_labor']) < 0.01
    assert abs(contribution - s['contribution']) < 0.02
    recalculated.append({'name': s['name'], 'proceeds': round(proceeds, 2), 'contribution': round(contribution, 2)})
base = next(s for s in finance['scenarios'] if s['name'] == 'Base')
base_paid_cpi_ceiling = (base['developer_proceeds'] - base['support_labor']) / base['installs']
assert round(base_paid_cpi_ceiling, 2) == 0.31
print({'scenarios': recalculated, 'base_paid_cpi_ceiling': round(base_paid_cpi_ceiling, 2)})

## Desk decision consistency

In [ ]:
blocked = any(gate['status'] == 'blocked' for gate in assessment['fatal_gates'])
unknown = any(gate['status'] == 'unknown' for gate in assessment['fatal_gates'])
grade = assessment['overall_evidence_grade']
opp = assessment['opportunity']
if blocked:
    expected_decision = 'ELENDIR'
elif unknown or grade == 'D':
    expected_decision = 'BELIRSIZ'
elif opp['high'] < 40:
    expected_decision = 'ELENDIR'
elif opp['low'] >= 60:
    expected_decision = 'TEST_ONCELIKLI'
else:
    expected_decision = 'TESTE_DEGER'
assert expected_decision == assessment['decision'] == 'TESTE_DEGER'
assert assessment['status'] == 'PREBUILD_PENDING'
print({'decision': expected_decision, 'status': assessment['status'], 'evidence_grade': grade})